In [8]:
# DTSC 2302 Final Project
# Charlotte Area Quality of Life Data

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.neighbors import KNeighborsRegressor, KNeighborsClassifier
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier

In [9]:
# Function to clean each QOL indicator file

def clean_qol_file(file_path, new_col_name):
    df = pd.read_csv(file_path)
    df = df[["NPA", "2023"]].copy()
    df = df.rename(columns={"2023": new_col_name})
    
    # remove $ signs, commas, and percent signs
    df[new_col_name] = (
        df[new_col_name]
        .astype(str)
        .str.replace("$", "", regex=False)
        .str.replace(",", "", regex=False)
        .str.replace("%", "", regex=False)
        .str.replace("N/A", "", regex=False)
    )
    
    df[new_col_name] = pd.to_numeric(df[new_col_name], errors="coerce")
    return df

In [16]:
# Load each dataset

income = clean_qol_file("/Users/aidancook/Downloads/Household Income.csv", "median_income")
# robust load for property_crime when '2023' may be missing
_path = "/Users/aidancook/Downloads/Crime - Property.csv"
_tmp = pd.read_csv(_path)

# find year-like columns (e.g., '2023', '2022', ...)
year_cols = [c for c in _tmp.columns if c.isdigit() and len(c) == 4]
if "2023" in _tmp.columns:
    year_col = "2023"
elif year_cols:
    year_col = str(max(int(c) for c in year_cols))
else:
    # fallback: pick the last column that isn't 'NPA'
    candidates = [c for c in _tmp.columns if c != "NPA"]
    year_col = candidates[-1] if candidates else None

if year_col is None:
    raise ValueError(f"No year column found in {_path}")

property_crime = _tmp[["NPA", year_col]].copy()
property_crime = property_crime.rename(columns={year_col: "property_crime"})

# clean values similarly to clean_qol_file
property_crime["property_crime"] = (
    property_crime["property_crime"]
    .astype(str)
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.replace("%", "", regex=False)
    .str.replace("N/A", "", regex=False)
    .str.replace("--", "", regex=False)
)

property_crime["property_crime"] = pd.to_numeric(property_crime["property_crime"], errors="coerce")

# Add more QOL files here
# robust load for age_of_death when '2023' may be missing
_path = "/Users/aidancook/Downloads/Age of Death.csv"
_tmp = pd.read_csv(_path)

# find year-like columns (e.g., '2023', '2022', ...)
year_cols = [c for c in _tmp.columns if c.isdigit() and len(c) == 4]
if "2023" in _tmp.columns:
    year_col = "2023"
elif year_cols:
    year_col = str(max(int(c) for c in year_cols))
else:
    # fallback: pick the last column that isn't 'NPA'
    candidates = [c for c in _tmp.columns if c != "NPA"]
    year_col = candidates[-1] if candidates else None

if year_col is None:
    raise ValueError(f"No year column found in {_path}")

age_of_death = _tmp[["NPA", year_col]].copy()
age_of_death = age_of_death.rename(columns={year_col: "age_of_death"})

age_of_death["age_of_death"] = (
    age_of_death["age_of_death"]
    .astype(str)
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.replace("%", "", regex=False)
    .str.replace("N/A", "", regex=False)
    .str.replace("--", "", regex=False)
)

age_of_death["age_of_death"] = pd.to_numeric(age_of_death["age_of_death"], errors="coerce")
unemployment = clean_qol_file("/Users/aidancook/Downloads/Employment.csv", "unemployment_rate")
education = clean_qol_file("/Users/aidancook/Downloads/Education Level - Bachelor's Degree (1).csv", "bachelors_degree_rate")
homeownership = clean_qol_file("/Users/aidancook/Downloads/Home Ownership.csv", "homeownership_rate")
# robust load for population_density when '2023' may be missing
_path = "/Users/aidancook/Downloads/Population Density.csv"
_tmp = pd.read_csv(_path)

# find year-like columns (e.g., '2023', '2022', ...)
year_cols = [c for c in _tmp.columns if c.isdigit() and len(c) == 4]
if "2023" in _tmp.columns:
    year_col = "2023"
elif year_cols:
    year_col = str(max(int(c) for c in year_cols))
else:
    # fallback: pick the last column that isn't 'NPA'
    candidates = [c for c in _tmp.columns if c != "NPA"]
    year_col = candidates[-1] if candidates else None

if year_col is None:
    raise ValueError(f"No year column found in {_path}")

population_density = _tmp[["NPA", year_col]].copy()
population_density = population_density.rename(columns={year_col: "population_density"})

population_density["population_density"] = (
    population_density["population_density"]
    .astype(str)
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.replace("%", "", regex=False)
    .str.replace("N/A", "", regex=False)
    .str.replace("--", "", regex=False)
)

population_density["population_density"] = pd.to_numeric(population_density["population_density"], errors="coerce")

In [18]:
# Merge all datasets by NPA

from functools import reduce

dfs = [
    income,
    property_crime,
    age_of_death,
    unemployment,
    education,
    homeownership,
    population_density
]

df = reduce(lambda left, right: pd.merge(left, right, on="NPA", how="inner"), dfs)

print(df.head())
print(df.shape)
print(df.columns)

   NPA  median_income  property_crime  age_of_death  unemployment_rate  \
0    2        75084.0            60.0          68.0               95.6   
1    3       117630.0            60.0          70.0               97.4   
2    4       250001.0             7.0           NaN               94.3   
3    5        49539.0            26.0          64.0               82.8   
4    6        37907.0            81.0          59.0              100.0   

   bachelors_degree_rate  homeownership_rate  population_density  
0                   35.2                40.3                   5  
1                   85.4                39.2                  10  
2                   89.4               100.0                   4  
3                    2.5                14.9                   5  
4                   21.0                37.3                   4  
(459, 8)
Index(['NPA', 'median_income', 'property_crime', 'age_of_death',
       'unemployment_rate', 'bachelors_degree_rate', 'homeownership_rate',
    

In [19]:
# Drop missing values

df = df.dropna()

print(df.shape)

(349, 8)


In [20]:
# Regression: predict median income

X = df[[
    "property_crime",
    "age_of_death",
    "unemployment_rate",
    "bachelors_degree_rate",
    "homeownership_rate",
    "population_density"
]]

y = df["median_income"]

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor
import numpy as np
import pandas as pd

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [21]:
# Regression models

reg_models = {
    "Linear Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LinearRegression())
    ]),
    
    "kNN Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", KNeighborsRegressor(n_neighbors=5))
    ]),
    
    "Random Forest Regression": RandomForestRegressor(
        n_estimators=500,
        random_state=42
    )
}

reg_results = []

for name, model in reg_models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    r2 = r2_score(y_test, preds)
    
    reg_results.append([name, rmse, r2])

reg_results_df = pd.DataFrame(reg_results, columns=["Model", "RMSE", "R2"])
print(reg_results_df)

                      Model          RMSE        R2
0         Linear Regression  18523.094841  0.758945
1            kNN Regression  18177.322791  0.767861
2  Random Forest Regression  17413.446353  0.786962


In [22]:
# Feature importance for random forest regression

rf_reg = reg_models["Random Forest Regression"]

importance_df = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rf_reg.feature_importances_
}).sort_values(by="Importance", ascending=False)

print(importance_df)

                 Feature  Importance
3  bachelors_degree_rate    0.658443
4     homeownership_rate    0.229093
0         property_crime    0.040179
1           age_of_death    0.032064
2      unemployment_rate    0.026143
5     population_density    0.014078


In [23]:
# Create classification target
# 1 = High Quality of Life, 0 = Low Quality of Life

df["High_QOL"] = np.where(
    df["median_income"] >= df["median_income"].median(),
    1,
    0
)

X_class = df[[
    "property_crime",
    "age_of_death",
    "unemployment_rate",
    "bachelors_degree_rate",
    "homeownership_rate",
    "population_density"
]]

y_class = df["High_QOL"]

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_class, y_class, test_size=0.2, random_state=42, stratify=y_class
)

In [24]:
# Classification models

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

class_models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=1000))
    ]),
    
    "kNN Classification": Pipeline([
        ("scaler", StandardScaler()),
        ("model", KNeighborsClassifier(n_neighbors=5))
    ]),
    
    "Random Forest Classification": RandomForestClassifier(
        n_estimators=500,
        random_state=42
    )
}

class_results = []

for name, model in class_models.items():
    model.fit(X_train_c, y_train_c)
    preds = model.predict(X_test_c)
    
    accuracy = accuracy_score(y_test_c, preds)
    precision = precision_score(y_test_c, preds, zero_division=0)
    recall = recall_score(y_test_c, preds, zero_division=0)
    f1 = f1_score(y_test_c, preds, zero_division=0)
    
    class_results.append([name, accuracy, precision, recall, f1])
    
    print("\n", name)
    print(confusion_matrix(y_test_c, preds))
    print(classification_report(y_test_c, preds, zero_division=0))

class_results_df = pd.DataFrame(
    class_results,
    columns=["Model", "Accuracy", "Precision", "Recall", "F1"]
)

print(class_results_df)


 Logistic Regression
[[32  3]
 [ 7 28]]
              precision    recall  f1-score   support

           0       0.82      0.91      0.86        35
           1       0.90      0.80      0.85        35

    accuracy                           0.86        70
   macro avg       0.86      0.86      0.86        70
weighted avg       0.86      0.86      0.86        70


 kNN Classification
[[33  2]
 [ 7 28]]
              precision    recall  f1-score   support

           0       0.82      0.94      0.88        35
           1       0.93      0.80      0.86        35

    accuracy                           0.87        70
   macro avg       0.88      0.87      0.87        70
weighted avg       0.88      0.87      0.87        70


 Random Forest Classification
[[32  3]
 [ 6 29]]
              precision    recall  f1-score   support

           0       0.84      0.91      0.88        35
           1       0.91      0.83      0.87        35

    accuracy                           0.87        

In [25]:
# Feature importance for random forest classification

rf_class = class_models["Random Forest Classification"]

class_importance_df = pd.DataFrame({
    "Feature": X_class.columns,
    "Importance": rf_class.feature_importances_
}).sort_values(by="Importance", ascending=False)

print(class_importance_df)

                 Feature  Importance
3  bachelors_degree_rate    0.423017
4     homeownership_rate    0.200975
0         property_crime    0.119334
1           age_of_death    0.114895
2      unemployment_rate    0.076026
5     population_density    0.065753
